# Data Retrieval Financials

| Field | Details |
|---|---|
| **Time window** | 1 Jun 2024 – 4 Nov 2024
| **Source** | `yfinance` (market data) · FRED API via `pandas_datareader` (macro indicators) |
| **Method** | Python API wrappers — daily close prices + quarterly macro series |
| **Market indicators** | S&P 500 · Crude oil (WTI) · VIX · 10-year US Treasury yield · USD Index |
| **Macro indicators** | Real disposable income · Real GDP · Unemployment rate · CPI inflation · Consumer sentiment |
| **Saved to** | `Data/1_Bronze/Financials/market.csv` · `Data/1_Bronze/Financials/macros.csv` |

### market.csv columns

| Column | Type | Description |
|---|---|---|
| `Date` | date | Trading day (YYYY-MM-DD) |
| `SP500` | float | S&P 500 closing price |
| `Oil` | float | WTI crude oil closing price (USD/barrel) |
| `VIX` | float | CBOE Volatility Index (fear gauge) |
| `TenYearBond` | float | 10-year US Treasury yield (%) |
| `USDIndex` | float | US Dollar Index (DXY) |

### macros.csv columns

| Column | Type | Description |
|---|---|---|
| `DATE` | date | Quarter start date |
| `RealDisposableIncome` | float | Real disposable personal income (index) |
| `RealGDP` | float | Real GDP growth rate (%) |
| `UnemploymentRate` | float | US unemployment rate (%) |
| `CPIInflation` | float | Consumer Price Index inflation rate (%) |
| `ConsumerSentiment` | float | University of Michigan Consumer Sentiment Index |

<!-- toc -->
## Contents
- [Setup](#setup)
- [1. Market Data](#1-market-data)
  - [market.csv columns](#marketcsv-columns)
- [2. Macro Data](#2-macro-data)
  - [macros.csv columns](#macroscsv-columns)


## Setup

In [ ]:
import pandas as pd
import yfinance as yf
from pandas_datareader import data as pdr

## 1. Market Data

### market.csv columns

| Column | Type | Description |
|---|---|---|
| `Date` | date | Trading day (YYYY-MM-DD) |
| `SP500` | float | S&P 500 closing price |
| `Oil` | float | WTI crude oil closing price (USD/barrel) |
| `VIX` | float | CBOE Volatility Index (fear gauge) |
| `TenYearBond` | float | 10-year US Treasury yield (%) |
| `USDIndex` | float | US Dollar Index (DXY) |

In [ ]:
start_date = '2024-06-01'  # Start 5 weeks before basetable to warm up rolling windows (30d max)
end_date   = '2024-11-05'  # yfinance end is exclusive, so use Nov 5 to include Nov 4

tickers = {
    '^GSPC': 'SP500',
    'CL=F': 'Oil',
    '^VIX': 'VIX',
    '^TNX': 'TenYearBond',
    'DX-Y.NYB': 'USDIndex'
}

df_list = []
for ticker, name in tickers.items():
    df = yf.download(ticker, start=start_date, end=end_date, progress=False)
    if 'Adj Close' in df.columns:
        df = df[['Adj Close']].rename(columns={'Adj Close': name})
    elif 'Close' in df.columns:
        df = df[['Close']].rename(columns={'Close': name})
    else:
        print(f"Warning: {ticker} has no price data.")
        continue
    df_list.append(df)

if df_list:
    market_df = pd.concat(df_list, axis=1)
    if isinstance(market_df.columns, pd.MultiIndex):
        market_df.columns = market_df.columns.get_level_values(0)
    market_df.to_csv('../Data/1_Bronze/Financials/market.csv', index=True)
    print('Market data saved to ../Data/1_Bronze/Financials/market.csv')
    print(market_df.tail())
else:
    print('No market data fetched.')

## 2. Macro Data

### macros.csv columns

| Column | Type | Description |
|---|---|---|
| `DATE` | date | Quarter start date |
| `RealDisposableIncome` | float | Real disposable personal income (index) |
| `RealGDP` | float | Real GDP growth rate (%) |
| `UnemploymentRate` | float | US unemployment rate (%) |
| `CPIInflation` | float | Consumer Price Index inflation rate (%) |
| `ConsumerSentiment` | float | University of Michigan Consumer Sentiment Index |

In [ ]:
fred_api_key = '981ac4d5555a2e7f975219ee2bc806a1'
start_date   = '2024-06-01'
end_date     = '2024-11-04'

series = {
    'DSPIC96': 'RealDisposableIncome',
    'GDPC1':   'RealGDP',
    'UNRATE':  'UnemploymentRate',
    'CPIAUCSL':'CPIInflation',
    'UMCSENT': 'ConsumerSentiment'
}

fred_data = {}
for symbol, name in series.items():
    try:
        fred_data[symbol] = pdr.DataReader(symbol, 'fred', start_date, end_date, api_key=fred_api_key)
        print(f"Fetched {symbol}: {fred_data[symbol].shape[0]} rows")
    except Exception as e:
        print(f"Error fetching {symbol}: {e}")

if fred_data:
    macros_df = pd.concat(fred_data.values(), axis=1)
    macros_df.columns = list(series.values())
    macros_df.to_csv('../Data/1_Bronze/Financials/macros.csv', index=True)
    print('Macros saved to ../Data/1_Bronze/Financials/macros.csv')
    print(macros_df.tail())
else:
    print('No macros data fetched.')